In [1]:
# 数据处理库
import sys
import pandas as pd
import numpy as np
from dataset_deduplicate_utils import*
from datetime import datetime

In [2]:
class Config:
    
    TIMESTAMP = datetime.now().strftime('%y%m%d') 

    IN_DIR = '../data/input'
    IN_OUT_DIR = '../data/in_out'
    OUT_DIR = '../data/output'
    TEMP_DIR = '../tmp_data'

    MISSING_DATA_MAG_DIR = os.path.join(TEMP_DIR, 'mis_data_mag')
    DEDUPLICATE_DIR = os.path.join(TEMP_DIR, 'deduplicate')   

    # 输入文件路径
    DATADET_WITH_IDS = os.path.join(IN_OUT_DIR, f'{TIMESTAMP}_raw-ds_add-ids_dd.csv')

    # 输出文件路径
    DATASET_DEDUPLICATED = os.path.join(IN_OUT_DIR, f'{TIMESTAMP}_ds-with-ids_dd_fg.csv')
    DATASET_DEDUPLICATED_OUT = os.path.join(OUT_DIR, f'{TIMESTAMP}_out_ds-dd.csv')

In [3]:
TIMESTAMP = datetime.now().strftime('%y%m%d') 
# 定义需要检查缺失值的列
essential_cols = ['c1', 'c2', 'x1', 'x2', 'beta', 'num']

# 【配置】 定义组分名称列 (使用 SMILES 作为唯一标识)
# 注意：这三列将用于生成指纹，并在排序中起决定性作用
str_cols = [
    'c1_ismiles',
    'c2_ismiles',
    'c3_ismiles'
]

# 【配置】 定义组分比例列
# 注意：这里只包含比例 (x)，不能包含温度 (t)
num_cols = [
    'x1',
    'x2', 
    'x3'
]
property_col = 'beta'

# 中间结果输出目录
interim_output_dir = Config.DEDUPLICATE_DIR

# 定义处理完后要删除的辅助列 (例如原始的 c1, c2 名称列，如果不再需要)
drop_columns = ['c1', 'c2', 'c3']

# =============================================================================
# 2. 主程序执行流程
# =============================================================================

if __name__ == "__main__":
    print("="*50)
    print(f"开始执行数据去重流程")
    print(f"工作目录: {os.getcwd()}")
    print("="*50)

    # 1. 读取文件
    if not os.path.exists(Config.DATADET_WITH_IDS):
        print(f"错误: 找不到输入文件 {Config.DATADET_WITH_IDS}")
        # 为了调试方便，如果没有文件，可以生成一个假数据测试 (可选)
        sys.exit(1)

    print(f"正在读取文件: {Config.DATADET_WITH_IDS}")
    try:
        # 读取数据 (推荐使用 openpyxl 引擎读取 xlsx)
        original_df = pd.read_csv(Config.DATADET_WITH_IDS)
    except Exception as e:
        print(f"读取 Excel 失败: {e}")
        sys.exit(1)

    # 2. 调用 utils 中的去重函数
    # 注意参数的对应关系
    result_df = process_dataset_deduplication(
        original_df=original_df,
        essential_cols=essential_cols,
        str_cols=str_cols,       # 传入组分标识列
        num_cols=num_cols,       # 传入比例列 
        property_col=property_col, # 目标属性 (vis)
        drop_columns=drop_columns,
        output_dir=interim_output_dir,
        ref_col='num'            # 参考文献列名
    )

    # 3. 生成质量报告
    print("\n正在生成数据质量报告...")
    qlty_sum_output_path = os.path.join(Config.MISSING_DATA_MAG_DIR, f'{TIMESTAMP}_ds-dd_qlty-sum.xlsx')
    
    try:
        data_quality_summary(result_df, qlty_sum_output_path)
    except Exception as e:
        print(f"生成质量报告时发生错误: {e}")

    # 4. 保存最终结果
    print("\n正在保存最终结果...")
    result_df.to_csv(Config.DATASET_DEDUPLICATED, index=False)
    result_df.to_csv(Config.DATASET_DEDUPLICATED_OUT, index=False)

    print("-" * 30)
    print("全部流程执行完毕。")
    print(f"最终结果文件: {Config.DATASET_DEDUPLICATED_OUT}")
    print("-" * 30)

开始执行数据去重流程
工作目录: d:\envs\codes\DES-DataAlchemy\dataset_pretraetment\src
正在读取文件: ../data/in_out\260307_raw-ds_add-ids_dd.csv
=== [Utils] 1. 缺失值处理 ===
未检测到关键列的缺失值。
剩余行数: 813

=== [Utils] 2. 生成规范化 Key (解决 A+B vs B+A 问题，不包含温度) ===
正在生成唯一标识键...

=== [Utils] 3. 识别重复数据 ===
涉及重复的总行数: 446
正在优化 Excel 格式 (合并重复组编号)...
重复数据详情已保存至: ../tmp_data\deduplicate\00_duplicated_data_details.xlsx (已合并单元格)

=== [Utils] 4. 合并重复数据 ===
已将 446 行重复数据压缩为 100 条唯一记录。

=== [Utils] 5. 生成最终数据集 ===
最终数据集构建完成。总行数: 467

正在生成数据质量报告...
质量报告已生成: ../tmp_data\mis_data_mag\260307_ds-dd_qlty-sum.txt

正在保存最终结果...
------------------------------
全部流程执行完毕。
最终结果文件: ../data/output\260307_out_ds-dd.csv
------------------------------
